<a href="https://colab.research.google.com/github/ChaimElchik/GPS-Demo/blob/main/GPS_DEM_DepthAnythingDistanceSamplingVideoSequenceMediumV4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Object Detection and GPS Localization Video Sequence Light


---
## Setup Environment

In [1]:
# Install necessary packages
!pip install piexif geopy pyproj torch torchvision transformers timm accelerate -q
!pip install ultralytics==8.3.18 --upgrade --quiet
!apt-get install -y exiftool -qq

# Standard library imports
import csv
import io
import json
import math
import os
import re
import subprocess
import time
import traceback
from datetime import datetime, timedelta
from pathlib import Path

# Third-party library imports
import cv2
import numpy as np
import pandas as pd
import requests
import torch
from geopy.distance import geodesic
from matplotlib import pyplot as plt
from PIL import Image
from pyproj import Transformer
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from ultralytics import YOLO

# Google Colab / IPython specific imports
from google.colab import files
from IPython.display import Image as IPImage, display

# Create output directories
os.makedirs("Detections", exist_ok=True)
os.makedirs("Processed_Output", exist_ok=True)

print("\nSetup Complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.6/876.6 kB 14.0 MB/s eta 0:00:00
Selecting previously unselected package libarchive-zip-perl.
(Reading database ... 126281 files and directories currently 


##  1. Upload Model, SRT File, Video File and Tracker Config File

In [2]:
print("--- Step 1: Upload Files ---")
print("Please upload your video, SRT file, your .pt model, and your tracker config (e.g., 'botsort.yaml').")

uploaded = files.upload()

video_path = None
srt_path = None
model_path = None
tracker_config_path = None

for fn in uploaded.keys():
    if fn.lower().endswith('.pt'):
        model_path = fn
        print(f"✅ Model file '{fn}' found.")
    elif fn.lower().endswith('.yaml'):
        tracker_config_path = fn
        print(f"✅ Tracker config file '{fn}' found.")
    elif fn.lower().endswith(('.mp4', '.mov', '.avi')):
        video_path = fn
        print(f"✅ Video file '{fn}' found.")
    elif fn.lower().endswith('.srt'):
        srt_path = fn
        print(f"✅ SRT file '{fn}' found.")

print("\n--- Verifying files ---")
if not video_path: print("❌ ERROR: Video file not uploaded.")
if not srt_path: print("❌ ERROR: SRT file not uploaded.")
if not model_path: print(f"❌ ERROR: Model .pt file not uploaded.")
if not tracker_config_path: print("❌ ERROR: Tracker config .yaml file not uploaded.")

if video_path and srt_path and model_path and tracker_config_path:
    MODEL_PATH = model_path
    print("\n--- All files ready for processing! ---")

--- Step 1: Upload Files ---
Please upload your video, SRT file, your .pt model, and your tracker config (e.g., 'botsort.yaml').


Saving best.pt to best.pt
Saving botsortV5.yaml to botsortV5.yaml
Saving DJI_20250618120033_0001_D.SRT to DJI_20250618120033_0001_D.SRT
Saving Video_X_Axis.MP4 to Video_X_Axis.MP4
✅ Model file 'best.pt' found.
✅ Tracker config file 'botsortV5.yaml' found.
✅ SRT file 'DJI_20250618120033_0001_D.SRT' found.
✅ Video file 'Video_X_Axis.MP4' found.

--- Verifying files ---

--- All files ready for processing! ---


---
## 2. Core Logic and Helper Functions


In [46]:
# --- Import Libraries ---
import os
import cv2
import numpy as np
import json
import re
import math
import time
import traceback
import csv
from pathlib import Path
import requests
import torch
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from PIL import Image
from collections import deque

# --- Dependencies Check ---
try:
    from ultralytics import YOLO
    from geopy.distance import geodesic
    from geopy.point import Point
except ImportError as e:
    print(f"ERROR: Missing dependency - {e}. Please install required libraries.")
    print("Run: pip install ultralytics opencv-python pyproj geopy requests torch torchvision transformers timm accelerate Pillow")
    exit()

# --- Advanced Kalman Filter Class ---
class AdvancedKalmanFilter:
    def __init__(self, dt, std_acc, initial_meas_noise):
        self.dt = dt
        self.x = np.zeros((6, 1))
        self.A = np.eye(6)
        self.A[0, 3] = self.A[1, 4] = self.A[2, 5] = dt
        self.H = np.eye(3, 6)
        self.Q = np.eye(6) * (std_acc**2)
        self.R = np.eye(3) * initial_meas_noise
        self.P = np.eye(6) * 1000

    def predict(self):
        self.x = np.dot(self.A, self.x)
        self.P = np.dot(np.dot(self.A, self.P), self.A.T) + self.Q
        return self.x

    def update(self, z):
        y = z - np.dot(self.H, self.x)
        S = np.dot(self.H, np.dot(self.P, self.H.T)) + self.R
        K = np.dot(np.dot(self.P, self.H.T), np.linalg.inv(S))
        self.x = self.x + np.dot(K, y)
        self.P = np.dot((np.eye(6) - np.dot(K, self.H)), self.P)
        return self.x

# --- Global Configuration & Sensor Setup ---
OUTPUT_DIR = "Video_Processing_Output_Medium"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEPTH_MODEL_NAME = 'depth-anything/Depth-Anything-V2-Large-hf'
SENSOR_WIDTH_MM = 17.3
SENSOR_HEIGHT_MM = 13.0
print(f"INFO: Using Sensor Size {SENSOR_WIDTH_MM}mm x {SENSOR_HEIGHT_MM} (Mavic 3 Pro Main Cam).")
print(f"INFO: Using device: {DEVICE} for deep learning models.")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Helper Functions ---
def parse_srt_file(srt_path):
    print(f"INFO: Parsing SRT file: {srt_path}")
    metadata_map = {}
    with open(srt_path, 'r', encoding='utf-8') as f: content = f.read()
    pattern = re.compile(
        r"FrameCnt: (\d+).*?\[focal_len: ([\d\.]+)\]"
        r".*?\[latitude: ([\d\.\-]+)\] \[longitude: ([\d\.\-]+)\] "
        r"\[rel_alt: ([\d\.\-]+) abs_alt: ([\d\.\-]+)\] "
        r"\[gb_yaw: ([\d\.\-]+) gb_pitch: ([\d\.\-]+) gb_roll: ([\d\.\-]+)\]", re.DOTALL)
    for match in pattern.finditer(content):
        frame_cnt = int(match.group(1))
        metadata_map[frame_cnt] = {
            'focal_len': float(match.group(2)), 'latitude': float(match.group(3)),
            'longitude': float(match.group(4)), 'rel_alt': float(match.group(5)),
            'abs_alt': float(match.group(6)), 'gb_yaw': float(match.group(7)),
            'gb_pitch': float(match.group(8)), 'gb_roll': float(match.group(9)),}
    print(f"✅ Successfully parsed metadata for {len(metadata_map)} frames from SRT.")
    return metadata_map

def get_depth_map(frame, model, processor):
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    prediction = torch.nn.functional.interpolate(
        outputs.predicted_depth.unsqueeze(1), size=image.size[::-1], mode="bicubic", align_corners=False)
    return prediction.squeeze().cpu().numpy()

def get_dem_elevation_from_api(latitude, longitude):
    try:
        url = f"https://api.opentopodata.org/v1/eudem25m?locations={latitude},{longitude}"
        response = requests.get(url, verify=False, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data['results'] and data['results'][0]['elevation'] is not None:
                return data['results'][0]['elevation']
    except requests.exceptions.RequestException: return None
    return None

def get_camera_intrinsics(f_mm, s_w_mm, s_h_mm, i_w, i_h):
    fx = i_w * f_mm / s_w_mm; fy = i_h * f_mm / s_h_mm
    cx, cy = i_w / 2, i_h / 2
    return np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])

def get_rotation_matrix(pitch_deg, yaw_deg, roll_deg):
    yaw, pitch, roll = map(math.radians, [yaw_deg, pitch_deg, roll_deg])
    Rz = np.array([[math.cos(yaw), -math.sin(yaw), 0], [math.sin(yaw), math.cos(yaw), 0], [0, 0, 1]])
    Ry = np.array([[math.cos(pitch), 0, math.sin(pitch)], [0, 1, 0], [-math.sin(pitch), 0, math.cos(pitch)]])
    Rx = np.array([[1, 0, 0], [0, math.cos(roll), -math.sin(roll)], [0, math.sin(roll), math.cos(roll)]])
    R_gimbal = Rz @ Ry @ Rx
    R_cam_to_body = np.array([[0, 1, 0], [0, 0, 1], [1, 0, 0]]).T
    return R_gimbal @ R_cam_to_body

def calculate_bearing(lat1, lon1, lat2, lon2):
    lat1_rad, lon1_rad = math.radians(lat1), math.radians(lon1)
    lat2_rad, lon2_rad = math.radians(lat2), math.radians(lon2)
    delta_lon = lon2_rad - lon1_rad
    y = math.sin(delta_lon) * math.cos(lat2_rad)
    x = math.cos(lat1_rad) * math.sin(lat2_rad) - math.sin(lat1_rad) * math.cos(lat2_rad) * math.cos(delta_lon)
    return math.degrees(math.atan2(y, x))

def ned_to_gps(origin_lat, origin_lon, ned_point):
    north, east, _ = ned_point
    bearing = math.degrees(math.atan2(east, north))
    distance_meters = math.hypot(east, north)
    destination = geodesic(meters=distance_meters).destination(Point(origin_lat, origin_lon), bearing)
    return destination.latitude, destination.longitude

def image_point_to_ned(u, v, K, R, abs_depth_map):
    v_idx, u_idx = int(round(v)), int(round(u))
    if not (0 <= v_idx < abs_depth_map.shape[0] and 0 <= u_idx < abs_depth_map.shape[1]): return None
    distance_to_target = abs_depth_map[v_idx, u_idx]
    K_inv = np.linalg.inv(K)
    ray_cam = K_inv @ np.array([u, v, 1])
    ray_cam_unit = ray_cam / np.linalg.norm(ray_cam)
    point_in_cam_coords = ray_cam_unit * distance_to_target
    ned_offsets = R @ point_in_cam_coords
    return ned_offsets

def draw_overlays(frame, tracked_objects_data):
    frame_height, frame_width, _ = frame.shape
    cv2.line(frame, (frame_width // 2, 0), (frame_width // 2, frame_height), (255, 0, 0), 2)
    for data in tracked_objects_data:
        x1, y1, x2, y2 = data['box']
        obj_id = data['id']; conf = data['conf']
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        text = f"ID: {obj_id} | Conf: {conf:.2f}"
        if 'gps' in data:
            lat, lon = data['gps']
            text += f" | GPS: {lat:.6f}, {lon:.6f}"
        cv2.putText(frame, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    return frame

def save_frame_by_frame_log(all_results):
    filepath = os.path.join(OUTPUT_DIR, 'frame_by_frame_log.csv')
    with open(filepath, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Frame', 'Timestamp', 'ObjectID', 'Latitude', 'Longitude', 'Confidence', 'center_u', 'center_v'])
        for result in all_results:
            writer.writerow([result['frame'], result['timestamp'], result['id'],
                f"{result['lat']:.6f}", f"{result['lon']:.6f}", f"{result['conf']:.4f}",
                f"{result['center_u']:.2f}", f"{result['center_v']:.2f}"])
    print(f"✅ Frame-by-frame log saved to: {filepath}")

def save_unique_objects_summary(all_results):
    filepath = os.path.join(OUTPUT_DIR, 'unique_objects_first_seen.csv')
    first_seen = {}
    for result in all_results:
        obj_id = result['id']
        if obj_id not in first_seen:
            first_seen[obj_id] = {'id': obj_id, 'timestamp': result['timestamp'],
                'lat': result['lat'], 'lon': result['lon'], 'conf': result['conf']}
    with open(filepath, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['ObjectID', 'TimestampFirstSeen', 'Latitude', 'Longitude', 'Confidence'])
        for obj_id in sorted(first_seen.keys()):
            data = first_seen[obj_id]
            writer.writerow([data['id'], data['timestamp'], f"{data['lat']:.6f}",
                f"{data['lon']:.6f}", f"{data['conf']:.4f}"])
    print(f"✅ Unique objects summary saved to: {filepath}")


INFO: Using Sensor Size 17.3mm x 13.0 (Mavic 3 Pro Main Cam).
INFO: Using device: cpu for deep learning models.


---
## 3. Main Execution Block

In [47]:
def main_fixed_interval_pipeline(video_path, srt_path, model_path, tracker_config):
    start_time = time.time()
    print("\n--- 🚀 Starting Fixed-Interval Processing Pipeline 🚀 ---")

    # --- Configuration ---
    FRAME_PROCESSING_INTERVAL = 15

    # Kalman Filter parameters
    STATIONARY_PROCESS_NOISE = 0.01
    MOVING_PROCESS_NOISE = 1.0
    MOVEMENT_CHI2_THRESHOLD = 16.27
    MEASUREMENT_HISTORY_LENGTH = 15

    try:
        yolo_model = YOLO(model_path)
        depth_processor = AutoImageProcessor.from_pretrained(DEPTH_MODEL_NAME)
        depth_model = AutoModelForDepthEstimation.from_pretrained(DEPTH_MODEL_NAME).to(DEVICE)
        srt_metadata = parse_srt_file(srt_path)
    except Exception as e:
        print(f"❌ FATAL ERROR during initialization: {e}")
        return

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ ERROR: Cannot open video file {video_path}")
        return

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"INFO: Video Properties: {frame_width}x{frame_height} @ {fps:.2f} FPS, {total_frames} total frames.")

    output_video_path = os.path.join(OUTPUT_DIR, 'annotated_video_fixed_interval.mp4')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_video = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))
    print(f"INFO: Output video will be saved to: {output_video_path}")

    # --- Trackers and Filters ---
    kalman_filters = {}
    measurement_history = {}

    frame_count = 0
    all_frame_results = []
    origin_lat, origin_lon = None, None

    while cap.isOpened():
        success, frame = cap.read()
        if not success: break

        frame_count += 1
        timestamp = frame_count / fps

        if frame_count not in srt_metadata:
            out_video.write(frame)
            continue
        meta = srt_metadata[frame_count]

        if origin_lat is None:
            origin_lat, origin_lon = meta['latitude'], meta['longitude']

        # --- Step 1: Track on EVERY frame ---
        results = yolo_model.track(frame, persist=True, tracker=tracker_config, conf=0.5, verbose=False)

        tracked_objects_data = []
        for result in results:
            if result.boxes.id is not None:
                boxes = result.boxes.xyxy.cpu().numpy().astype(int)
                ids = result.boxes.id.cpu().numpy().astype(int)
                confs = result.boxes.conf.cpu().numpy()
                for i in range(len(ids)):
                    tracked_objects_data.append({'id': ids[i], 'box': boxes[i], 'conf': confs[i]})

        if not tracked_objects_data:
            out_video.write(frame)
            continue

        # --- Step 2: Decide whether to perform a full update or just predict ---
        is_anchor_frame = (frame_count % FRAME_PROCESSING_INTERVAL == 0)

        if is_anchor_frame:
            print(f"\n--- Anchor Frame {frame_count}: Performing Full Geolocation Update ---")
            try:
                # --- This is the full, expensive calculation block ---
                ground_elevation = get_dem_elevation_from_api(meta['latitude'], meta['longitude'])
                base_agl = (meta['abs_alt'] - ground_elevation) if ground_elevation is not None else meta['rel_alt']
                relative_depth_map = get_depth_map(frame, depth_model, depth_processor)
                center_pixel_depth = relative_depth_map[frame_height // 2, frame_width // 2]
                scale_factor = base_agl / center_pixel_depth if center_pixel_depth > 1e-6 else 1.0
                absolute_depth_map = relative_depth_map * scale_factor

                K = get_camera_intrinsics(meta['focal_len'], SENSOR_WIDTH_MM, SENSOR_HEIGHT_MM, frame_width, frame_height)
                R = get_rotation_matrix(meta['gb_pitch'], meta['gb_yaw'], meta['gb_roll'])

                for obj_data in tracked_objects_data:
                    center_u, center_v = (obj_data['box'][0] + obj_data['box'][2]) / 2, (obj_data['box'][1] + obj_data['box'][3]) / 2
                    ned_offset = image_point_to_ned(center_u, center_v, K, R, absolute_depth_map)

                    if ned_offset is not None:
                        lat, lon = calculate_destination_gps(meta['latitude'], meta['longitude'], ned_offset[1], ned_offset[0])
                        obj_id = obj_data['id']

                        if lat is not None:
                            if obj_id not in kalman_filters:
                                kalman_filters[obj_id] = AdvancedKalmanFilter(1/fps, std_acc=STATIONARY_PROCESS_NOISE, initial_meas_noise=1.0)
                                kalman_filters[obj_id].x[0], kalman_filters[obj_id].x[1] = lat, lon

                            # Update the filter with the new measurement
                            smoothed_state = kalman_filters[obj_id].update(np.array([[lat], [lon], [base_agl]]))
                            smoothed_lat, smoothed_lon = smoothed_state[0,0], smoothed_state[1,0]

                            obj_data['gps'] = (smoothed_lat, smoothed_lon)
                            all_frame_results.append({'frame': frame_count, 'timestamp': f"{timestamp:.3f}", 'id': obj_id,
                                'lat': smoothed_lat, 'lon': smoothed_lon, 'conf': obj_data['conf'],
                                'center_u': center_u, 'center_v': center_v})
            except Exception as e:
                print(f"❌ ERROR on anchor frame {frame_count}: {e}")

        else: # This is an intermediate frame
            print(f"Frame {frame_count}: Predicting positions...")
            for obj_data in tracked_objects_data:
                obj_id = obj_data['id']
                if obj_id in kalman_filters:
                    # --- Just predict based on the last known state ---
                    predicted_state = kalman_filters[obj_id].predict()
                    predicted_lat, predicted_lon = predicted_state[0,0], predicted_state[1,0]

                    obj_data['gps'] = (predicted_lat, predicted_lon)
                    all_frame_results.append({'frame': frame_count, 'timestamp': f"{timestamp:.3f}", 'id': obj_id,
                        'lat': predicted_lat, 'lon': predicted_lon, 'conf': obj_data['conf'],
                        'center_u': (obj_data['box'][0] + obj_data['box'][2]) / 2,
                        'center_v': (obj_data['box'][1] + obj_data['box'][3]) / 2})

        annotated_frame = draw_overlays(frame.copy(), tracked_objects_data)
        out_video.write(annotated_frame)

    cap.release()
    out_video.release()
    print("\n\n--- ✅ Fixed-Interval Video Processing Complete ---")

    if all_frame_results:
        save_frame_by_frame_log(all_frame_results)
        save_unique_objects_summary(all_frame_results)
        print(f"✅ Annotated video saved to: {output_video_path}")
    else:
        print("INFO: No objects were successfully geolocated in the video.")

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"\nTotal process completed in {elapsed_time:.2f} seconds.")
    if total_frames > 0:
        print(f"Average time per frame: {elapsed_time / total_frames:.3f} seconds.")

if 'video_path' in locals() and video_path and 'srt_path' in locals() and srt_path and 'model_path' in locals() and model_path and 'tracker_config_path' in locals() and tracker_config_path:
    main_fixed_interval_pipeline(video_path, srt_path, model_path, tracker_config_path)
else:
    print("\n❌ Please run Cell 1 to upload all required files before running this cell.")


--- 🚀 Starting Fixed-Interval Processing Pipeline 🚀 ---
INFO: Parsing SRT file: DJI_20250618120033_0001_D.SRT
✅ Successfully parsed metadata for 1931 frames from SRT.
INFO: Video Properties: 1920x1080 @ 29.97 FPS, 449 total frames.
INFO: Output video will be saved to: Video_Processing_Output_Medium/annotated_video_fixed_interval.mp4
Frame 1: Predicting positions...
Frame 2: Predicting positions...
Frame 3: Predicting positions...
Frame 4: Predicting positions...
Frame 5: Predicting positions...
Frame 6: Predicting positions...
Frame 7: Predicting positions...
Frame 8: Predicting positions...
Frame 9: Predicting positions...
Frame 10: Predicting positions...
Frame 11: Predicting positions...
Frame 12: Predicting positions...
Frame 13: Predicting positions...
Frame 14: Predicting positions...

--- Anchor Frame 15: Performing Full Geolocation Update ---


/usr/local/lib/python3.11/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.opentopodata.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Frame 17: Predicting positions...
Frame 18: Predicting positions...
Frame 19: Predicting positions...
Frame 20: Predicting positions...
Frame 21: Predicting positions...
Frame 23: Predicting positions...
Frame 25: Predicting positions...
Frame 26: Predicting positions...
Frame 27: Predicting positions...
Frame 31: Predicting positions...
Frame 32: Predicting positions...
Frame 33: Predicting positions...
Frame 36: Predicting positions...
Frame 37: Predicting positions...
Frame 38: Predicting positions...
Frame 41: Predicting positions...
Frame 42: Predicting positions...
Frame 52: Predicting positions...


KeyboardInterrupt: 

---


## 4.

In [42]:
import pandas as pd
import os
from geopy.distance import geodesic
import numpy as np

# Define the path to the detailed log file generated by the Full Processing version
log_path = os.path.join('Video_Processing_Output_Full', 'frame_by_frame_log.csv')

print(f"--- Loading detailed log from: {log_path} ---")

if os.path.exists(log_path):
    try:
        # Load the detailed log into a DataFrame
        df_log = pd.read_csv(log_path)
        print("✅ Successfully loaded the frame-by-frame log.")

        # --- Intermittent Movement Calculation (for average) ---
        print("\n--- Calculating object movement between frames... ---")

        # Sort the data by object ID and then by timestamp to ensure correct order
        df_log = df_log.sort_values(by=['ObjectID', 'Timestamp'])

        movement_records = []

        # Group by each object ID to process its path individually
        for object_id, group in df_log.groupby('ObjectID'):
            # Get the previous location for each row by shifting the coordinate columns
            group['Prev_Latitude'] = group['Latitude'].shift(1)
            group['Prev_Longitude'] = group['Longitude'].shift(1)

            # Iterate through the path of the object
            for index, row in group.iterrows():
                # Skip the very first appearance of the object (as there's no previous point)
                if pd.notna(row['Prev_Latitude']):

                    # Define the current and previous GPS points
                    current_pos = (row['Latitude'], row['Longitude'])
                    prev_pos = (row['Prev_Latitude'], row['Prev_Longitude'])

                    # Calculate the distance moved in meters
                    distance_moved = geodesic(prev_pos, current_pos).meters

                    # Only log significant movements to keep the log clean
                    if distance_moved > 0.01: # Threshold to ignore tiny GPS jitter
                        movement_records.append({
                            'ObjectID': row['ObjectID'],
                            'DistanceMovedMeters': distance_moved
                        })

        if not df_log.empty:
            # --- Calculate Total Distance using Farthest Point ---
            print("\n--- Calculating summary statistics... ---")

            total_distance_records = []
            for object_id, group in df_log.groupby('ObjectID'):
                if len(group) > 1:
                    first_row = group.iloc[0]
                    start_pos = (first_row['Latitude'], first_row['Longitude'])

                    max_distance = 0
                    frame_at_max_distance = first_row['Frame']

                    # Iterate through the rest of the points in the object's path
                    for index, row in group.iloc[1:].iterrows():
                        current_pos = (row['Latitude'], row['Longitude'])
                        distance = geodesic(start_pos, current_pos).meters
                        # If this distance is the largest so far, update max_distance and the frame number
                        if distance > max_distance:
                            max_distance = distance
                            frame_at_max_distance = row['Frame']

                    total_distance_records.append({
                        'ObjectID': object_id,
                        'TotalDistanceMeters': max_distance,
                        'FrameAtMaxDistance': frame_at_max_distance
                    })

            df_total_dist = pd.DataFrame(total_distance_records)

            # --- Calculate the average of the intermittent moves ---
            if movement_records:
                 df_movement = pd.DataFrame(movement_records)
                 df_avg_move = df_movement.groupby('ObjectID')['DistanceMovedMeters'].mean().reset_index()
                 df_avg_move.rename(columns={'AverageMoveMeters': 'AverageMoveMeters'}, inplace=True)

                 # Merge the total distance and average move summaries
                 df_summary = pd.merge(df_total_dist, df_avg_move, on='ObjectID', how='left')
            else:
                # If there were no intermittent moves, just use the total distance df
                df_summary = df_total_dist
                df_summary['AverageMoveMeters'] = 0.0

            if not df_summary.empty:
                print("\n--- Summary of Movement per Object ---")
                display(df_summary.set_index('ObjectID'))
            else:
                print("\n-> No objects with multiple detections were found to analyze for movement summary.")

        else:
            print("\n-> No object detections were logged to analyze.")

    except Exception as e:
        print(f"❌ An error occurred during movement calculation: {e}")
        traceback.print_exc()
else:
    print(f"❌ ERROR: The detailed log file was not found at '{log_path}'.")
    print("Please ensure the main processing cell (CELL 3) completed successfully and generated the log.")


--- Loading results from: Video_Processing_Output_Medium/unique_objects_first_seen.csv ---
✅ Successfully loaded the summary of unique object detections:


,ObjectID,TimestampFirstSeen,Latitude,Longitude,Confidence
0,1,0.033,52.725881,-2.746657,0.7774


In [43]:
import pandas as pd
import os
from geopy.distance import geodesic

# Define the path to the detailed log file
log_path = os.path.join('Video_Processing_Output_Medium', 'frame_by_frame_log.csv')

print(f"--- Loading detailed log from: {log_path} ---")

if os.path.exists(log_path):
    try:
        # Load the detailed log into a DataFrame
        df_log = pd.read_csv(log_path)
        print("✅ Successfully loaded the frame-by-frame log.")

        # --- Intermittent Movement Calculation (for average) ---
        print("\n--- Calculating object movement between frames... ---")

        # Sort the data by object ID and then by timestamp to ensure correct order
        df_log = df_log.sort_values(by=['ObjectID', 'Timestamp'])

        movement_records = []

        # Group by each object ID to process its path individually
        for object_id, group in df_log.groupby('ObjectID'):
            group['Prev_Latitude'] = group['Latitude'].shift(1)
            group['Prev_Longitude'] = group['Longitude'].shift(1)

            for index, row in group.iterrows():
                if pd.notna(row['Prev_Latitude']):
                    current_pos = (row['Latitude'], row['Longitude'])
                    prev_pos = (row['Prev_Latitude'], row['Prev_Longitude'])
                    distance_moved = geodesic(prev_pos, current_pos).meters
                    if distance_moved > 0.01:
                        movement_records.append({
                            'ObjectID': row['ObjectID'],
                            'DistanceMovedMeters': distance_moved
                        })

        if movement_records:
            df_movement = pd.DataFrame(movement_records)
            print("\n✅ Intermittent movement calculation complete.")

            # --- MODIFIED: Calculate Total Distance using Farthest Point ---
            print("\n--- Calculating summary statistics... ---")

            total_distance_records = []
            for object_id, group in df_log.groupby('ObjectID'):
                if len(group) > 1:
                    first_row = group.iloc[0]
                    start_pos = (first_row['Latitude'], first_row['Longitude'])

                    max_distance = 0
                    # --- NEW: Variable to store the frame at max distance ---
                    frame_at_max_distance = first_row['Frame']

                    # Iterate through the rest of the points in the object's path
                    for index, row in group.iloc[1:].iterrows():
                        current_pos = (row['Latitude'], row['Longitude'])
                        # Calculate distance from the start to the current point
                        distance = geodesic(start_pos, current_pos).meters
                        # If this distance is the largest so far, update max_distance and the frame number
                        if distance > max_distance:
                            max_distance = distance
                            # --- NEW: Update the frame number ---
                            frame_at_max_distance = row['Frame']

                    total_distance_records.append({
                        'ObjectID': object_id,
                        'TotalDistanceMeters': max_distance,
                        # --- NEW: Add the frame number to the record ---
                        'FrameAtMaxDistance': frame_at_max_distance
                    })

            df_total_dist = pd.DataFrame(total_distance_records)

            # 2. Calculate the average of the intermittent moves
            df_avg_move = df_movement.groupby('ObjectID')['DistanceMovedMeters'].mean().reset_index()
            df_avg_move.rename(columns={'DistanceMovedMeters': 'AverageMoveMeters'}, inplace=True)

            # 3. Merge the total distance and average move summaries
            if not df_total_dist.empty:
                df_summary = pd.merge(df_total_dist, df_avg_move, on='ObjectID', how='left')
                print("\n--- Summary of Movement per Object ---")
                display(df_summary.set_index('ObjectID'))
            else:
                 print("\n--- Summary of Movement per Object (No total distance calculated) ---")
                 display(df_avg_move.set_index('ObjectID'))

        else:
            print("\n-> No significant object movement was detected between frames.")

    except Exception as e:
        print(f"❌ An error occurred during movement calculation: {e}")
        traceback.print_exc()
else:
    print(f"❌ ERROR: The detailed log file was not found at '{log_path}'.")
    print("Please ensure the main processing cell (CELL 3) completed successfully and generated the log.")


--- Loading detailed log from: Video_Processing_Output_Medium/frame_by_frame_log.csv ---
✅ Successfully loaded the frame-by-frame log.

--- Calculating object movement between frames... ---

✅ Intermittent movement calculation complete.

--- Calculating summary statistics... ---

--- Summary of Movement per Object ---


,TotalDistanceMeters,FrameAtMaxDistance,AverageMoveMeters
ObjectID,,,
1,18.263551,91.0,0.412818


In [37]:
import pandas as pd
import os
import requests
import cv2
from pathlib import Path # Import Path for the fallback

print("--- Starting Distance Sampling Analysis ---")

# --- Configuration ---
# Define paths to the files generated by the main process
log_path = os.path.join('Video_Processing_Output_Medium', 'frame_by_frame_log.csv')
video_path_for_props = video_path # Use the global video_path from Cell 1
srt_path_for_meta = srt_path # Use the global srt_path from Cell 1
output_csv_path = os.path.join('Video_Processing_Output_Medium', 'distance_sampling_data.csv')

def get_region_from_gps_once(lat, lon):
    """Performs a single reverse geocoding lookup."""
    print(f"\nINFO: Querying OpenStreetMap API for region name at {lat:.4f}, {lon:.4f}...")
    headers = {'User-Agent': 'EcologicalSurveyScript/1.0'}
    url = f"https://nominatim.openstreetmap.org/reverse?format=json&lat={lat}&lon={lon}&zoom=10"
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            data = response.json()
            address = data.get('address', {})
            name_parts = [
                address.get('nature_reserve'), address.get('national_park'),
                address.get('village'), address.get('town'), address.get('city'),
                address.get('state'), address.get('country')
            ]
            region_name = ', '.join(part for part in name_parts if part)
            if region_name:
                print(f"INFO: API lookup successful. Region: {region_name}")
                return region_name
    except requests.exceptions.RequestException as e:
        print(f"WARNING: Region API request failed. Error: {e}")
    return None

if os.path.exists(log_path):
    try:
        # Load the detailed log and SRT data
        df_log = pd.read_csv(log_path)

        # --- NEW: Verification Step ---
        required_columns = ['Latitude', 'Longitude', 'Frame', 'ObjectID', 'center_u']
        if not all(col in df_log.columns for col in required_columns):
            print(f"❌ ERROR: The log file is missing required columns. Found: {df_log.columns.to_list()}")
            print(f"Please re-run the main processing cell (Cell 3) to regenerate the log file correctly.")
        else:
            print("✅ Log file loaded successfully with all required columns.")
            srt_data = parse_srt_file(srt_path_for_meta)

            # Get video properties to find frame width/height
            cap = cv2.VideoCapture(video_path_for_props)
            frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            cap.release()

            # Get a single region label for the whole video from the first frame's coordinates
            # --- CORRECTED: Use 'Latitude' and 'Longitude' ---
            first_lat, first_lon = df_log.iloc[0]['Latitude'], df_log.iloc[0]['Longitude']
            region_label = get_region_from_gps_once(first_lat, first_lon)
            if not region_label:
                region_label = Path(video_path_for_props).stem # Fallback to video name

            # --- Main Calculation Loop ---
            distance_sampling_records = []
            for index, row in df_log.iterrows():
                # --- CORRECTED: Use 'Frame' ---
                frame_num = row['Frame']
                if frame_num in srt_data:
                    meta = srt_data[frame_num]

                    # Calculate Ground Sampling Distance (GSD) for this frame
                    gsd_w = (meta['rel_alt'] * SENSOR_WIDTH_MM) / (meta['focal_len'] * frame_width)
                    gsd_h = (meta['rel_alt'] * SENSOR_HEIGHT_MM) / (meta['focal_len'] * frame_height)

                    # --- CORRECTED: Use 'center_u' ---
                    pixel_distance = abs(row['center_u'] - (frame_width / 2))
                    meter_distance = pixel_distance * gsd_w

                    # Calculate ground coverage area and effort
                    ground_width_m = frame_width * gsd_w
                    ground_height_m = frame_height * gsd_h
                    area_sq_km = (ground_width_m * ground_height_m) / 1_000_000

                    # Append record in the desired format
                    # --- CORRECTED: Use 'ObjectID', 'Latitude', 'Longitude' ---
                    distance_sampling_records.append({
                        'Region.Label': region_label,
                        'Area.km2': f"{area_sq_km:.6f}",
                        'Sample.Label': f"frame_{frame_num}",
                        'Effort.m': f"{ground_height_m:.2f}",
                        'object': row['ObjectID'],
                        'distance': f"{meter_distance:.2f}",
                        'size': 1, # Assuming size of each detection is 1
                        'Lat': f"{row['Latitude']:.6f}",
                        'Lon': f"{row['Longitude']:.6f}"
                    })

            # --- Save the final CSV ---
            if distance_sampling_records:
                df_dist_sample = pd.DataFrame(distance_sampling_records)
                df_dist_sample.to_csv(output_csv_path, index=False)
                print(f"\n✅ Distance sampling data successfully saved to: {output_csv_path}")
                print("\n--- First 5 rows of the distance sampling data ---")
                display(df_dist_sample.head())
            else:
                print("\n-> No data to process for distance sampling.")

    except Exception as e:
        print(f"❌ An error occurred during distance sampling analysis: {e}")
        traceback.print_exc()
else:
    print(f"❌ ERROR: The detailed log file was not found at '{log_path}'.")
    print("Please ensure the main processing cell (CELL 3) completed successfully.")


--- Starting Distance Sampling Analysis ---
✅ Log file loaded successfully with all required columns.
INFO: Parsing SRT file: DJI_20250618120033_0001_D.SRT
✅ Successfully parsed metadata for 1931 frames from SRT.

INFO: Querying OpenStreetMap API for region name at 52.7258, -2.7466...
INFO: API lookup successful. Region: England, United Kingdom

✅ Distance sampling data successfully saved to: Video_Processing_Output_Medium/distance_sampling_data.csv

--- First 5 rows of the distance sampling data ---


,Region.Label,Area.km2,Sample.Label,Effort.m,object,distance,size,Lat,Lon
0,"England, United Kingdom",0.001253,frame_1.0,30.69,1.0,1.72,1,52.725849,-2.746646
1,"England, United Kingdom",0.001254,frame_2.0,30.69,1.0,1.70,1,52.725848,-2.746646
2,"England, United Kingdom",0.001254,frame_3.0,30.69,1.0,1.71,1,52.725847,-2.746646
3,"England, United Kingdom",0.001254,frame_4.0,30.69,1.0,1.73,1,52.725845,-2.746647
4,"England, United Kingdom",0.001254,frame_5.0,30.69,1.0,1.72,1,52.725844,-2.746647
